In [ ]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
COMMIT = '31e974b23b7e73b2f48b080b36865608efb879b8'
ROOT = Path('/content/e05_source')
SETUP_ENV = dict(os.environ, UV_PYTHON_INSTALL_DIR='/content/e05_python', UV_CACHE_DIR='/content/e05_uv_cache', UV_NATIVE_TLS='true', CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='1', MPLCONFIGDIR='/content/e05_mpl', MPLBACKEND='Agg')
print(subprocess.check_output(['nvidia-smi'], text=True))
assert not ROOT.exists()
subprocess.run(['git','clone','--quiet','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(ROOT)],check=True)
subprocess.run(['git','checkout','--quiet','--detach',COMMIT],cwd=ROOT,check=True)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip()
source = {p.relative_to(ROOT).as_posix():p.read_text(encoding='utf-8') for folder in ('src','configs','experiments/configs') for p in (ROOT/folder).rglob('*') if p.suffix in {'.py','.json'}}
code_hash = hashlib.sha256(json.dumps(source,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
assert code_hash == 'cdea42adb930904fb0cefcd282d784bb3ac8caa311d3f1e12ef568622f9c12d6'
subprocess.run([sys.executable,'-m','pip','install','--target','/content/e05_tools','uv==0.12.7'],env=SETUP_ENV,check=True)
UV = Path('/content/e05_tools/bin/uv')
subprocess.run([str(UV),'python','install','3.11.14'],env=SETUP_ENV,check=True)
subprocess.run([str(UV),'venv','--seed','--python','3.11.14','/content/e05_venv'],env=SETUP_ENV,check=True)
PY = Path('/content/e05_venv/bin/python')
freeze = (ROOT/'docs/qualifications/evidence/rayyan_requirements_freeze.txt').read_text().splitlines()
lock = Path('/content/e05_lock.txt')
lock.write_text('\n'.join(x for x in freeze if not x.startswith('-e '))+'\n')
r = subprocess.run([str(UV),'pip','install','--python',str(PY),'torch==2.5.1+cu124','--index-url','https://download.pytorch.org/whl/cu124'],env=SETUP_ENV,capture_output=True,text=True); print(r.stdout,r.stderr[-1500:]); r.check_returncode()
r = subprocess.run([str(UV),'pip','install','--python',str(PY),'-r',str(lock)],env=SETUP_ENV,capture_output=True,text=True); print(r.stdout,r.stderr[-1500:]); r.check_returncode()
subprocess.run([str(UV),'pip','install','--python',str(PY),'-e',str(ROOT),'--no-deps'],env=SETUP_ENV,check=True)
subprocess.run([str(PY),'-m','pip','check'],env=SETUP_ENV,check=True)
print('E05 isolated environment provisioned:',COMMIT,code_hash,flush=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import uuid, json
assert Path('/content/drive/MyDrive').is_dir()
STORE = Path('/content/drive/MyDrive/Nano-Language-Model/E05_seed42') / ('pilot_'+uuid.uuid4().hex)
STORE.mkdir(parents=True,exist_ok=False)
OUT = Path('/content/e05_pilot')
print('Fresh E05 storage:',STORE)

In [ ]:
DRIVE_QUOTA_FREE_BYTES = None  # Read actual Drive account quota before each new run.
assert isinstance(DRIVE_QUOTA_FREE_BYTES, int) and DRIVE_QUOTA_FREE_BYTES > 0
ORCH = Path('/content/e05_orchestration')
ORCH_COMMIT = '3aaaa911c69d06e2c9693f988edc8cb5b536abd6'
assert not ORCH.exists()
subprocess.run(['git','clone','--quiet','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(ORCH)],check=True)
subprocess.run(['git','checkout','--quiet','--detach',ORCH_COMMIT],cwd=ORCH,check=True)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ORCH,text=True).strip()
SETUP_ENV['PYTHONPATH'] = str(ORCH)
PREFLIGHT_CMD = [str(PY),str(ORCH/'scripts/pilot/e05_colab.py'),'preflight','--root',str(ROOT),'--output',str(OUT),'--store',str(STORE),'--quota-free-bytes',str(DRIVE_QUOTA_FREE_BYTES)]
preflight_log = (STORE/'preflight.log').open('x')
preflight_proc = subprocess.Popen(PREFLIGHT_CMD,cwd=ROOT,env=SETUP_ENV,stdout=preflight_log,stderr=subprocess.STDOUT,start_new_session=True)
print('E05 CUDA/data/storage preflight PID:',preflight_proc.pid)

In [ ]:
print('Preflight returncode:',preflight_proc.poll())
print((STORE/'preflight.log').read_text()[-2500:])
print('Qualification files:',[p.name for p in OUT.glob('*.json')])
print((OUT/'tests.json').read_text() if (OUT/'tests.json').exists() else 'Tests pending')
print((OUT/'activation_test.json').read_text() if (OUT/'activation_test.json').exists() else 'Activation test pending')
print((OUT/'preflight.json').read_text() if (OUT/'preflight.json').exists() else 'Storage qualification pending')

In [ ]:
RESEARCH_AUTHORIZED = False  # Set only for a separately approved execution.
assert RESEARCH_AUTHORIZED, 'Research execution requires explicit approval'
assert preflight_proc.poll() == 0
assert json.loads((OUT/'preflight.json').read_text())['status'] == 'passed'
assert not (OUT/'runs').exists() and not (STORE/'research_snapshots').exists()
RESEARCH_CMD = [str(PY),str(ORCH/'scripts/pilot/e05_colab.py'),'research','--root',str(ROOT),'--output',str(OUT),'--store',str(STORE)]
research_log = (STORE/'research.log').open('x')
research_proc = subprocess.Popen(RESEARCH_CMD,cwd=ROOT,env=SETUP_ENV,stdout=research_log,stderr=subprocess.STDOUT,start_new_session=True)
print('E05 seed42 research PID:',research_proc.pid,'authorized stop: 5000 completed updates')

In [ ]:
runs = list((OUT/'runs').iterdir()) if (OUT/'runs').exists() else []
run_dir = runs[0] if len(runs) == 1 else None
events = [json.loads(x) for x in (run_dir/'events.jsonl').read_text().splitlines()] if run_dir and (run_dir/'events.jsonl').exists() else []
metrics = [e['metric'] for e in events if e.get('event') == 'step']
validations = [m for m in metrics if 'validation' in m]
snaps = [p for p in (STORE/'research_snapshots').glob('*') if (p/'export_manifest.json').exists()]
print(json.dumps({'returncode':research_proc.poll(),'run_id':run_dir.name if run_dir else None,'completed_step':metrics[-1]['step'] if metrics else 0,'full_validations':len(validations),'latest_validation':validations[-1] if validations else None,'durable_snapshots':len(snaps),'failures':[e for e in events if e.get('event')=='failure'],'log_tail':(STORE/'research.log').read_text()[-700:]},indent=2))

In [ ]:
assert research_proc.poll() == 0, 'Wait for completed 5000-update research run'
assert json.loads((OUT/'completion.json').read_text())['status'] == '5000_completed_optimizer_updates'
POST = Path('/content/e05_postprocess_source')
POST_COMMIT = '6004b26a131bcb7a3558cdb15507a22f6ec0dfcf'
assert not POST.exists()
subprocess.run(['git','clone','--quiet','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(POST)],check=True)
subprocess.run(['git','checkout','--quiet','--detach',POST_COMMIT],cwd=POST,check=True)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=POST,text=True).strip()
FINISH_CMD = [str(PY),str(POST/'scripts/pilot/e05_colab.py'),'finish','--root',str(ROOT),'--output',str(OUT),'--store',str(STORE)]
finish_log = (STORE/'finish.log').open('x')
finish_proc = subprocess.Popen(FINISH_CMD,cwd=ROOT,env=SETUP_ENV,stdout=finish_log,stderr=subprocess.STDOUT,start_new_session=True)
print('Fresh CUDA recovery, full generation, archive publication PID:',finish_proc.pid,'additional optimizer updates: 0')
